# 02 - Emotion/control vector layer sweep (Phase 2)

Loads the validation report produced by `scripts/validate_vectors.py` and shows
held-out AUROC per layer for each emotion / control concept.

Run first:
```
python scripts/build_vectors.py --methods diff_of_means pca logistic lda
python scripts/validate_vectors.py
```

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
report = json.loads((ROOT / 'outputs/reports/vector_validation.json').read_text(encoding='utf-8'))
print('model:', report['model_id'], '| method:', report['method'], '| layers:', report['n_layers'])

In [ ]:
# best layer + AUROC per concept
for c, r in sorted(report['concepts'].items(), key=lambda kv: -(kv[1]['best_auroc'] or 0)):
    print(f"{c:24} {r['kind']:8} best_L={r['best_layer']:>3} AUROC={r['best_auroc']:.3f} d={r['best_cohens_d']:.2f}")

In [ ]:
# layer-sweep curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,5))
for c, r in report['concepts'].items():
    aur = [p['auroc'] or 0 for p in r['layer_sweep']]
    ax.plot(range(len(aur)), aur, '-' if r['kind']=='emotion' else '--', label=c, alpha=0.8)
ax.axhline(0.5, color='gray', ls=':')
ax.set_xlabel('layer'); ax.set_ylabel('held-out AUROC'); ax.legend(fontsize=7, ncol=2)
plt.show()

In [ ]:
# cross-concept collinearity (emotion vs control) at each best layer
coll = report.get('collinearity_best_layer', {})
top = sorted(coll.items(), key=lambda kv: -abs(kv[1]))[:15]
for k, v in top:
    print(f"{k:44} cos={v:+.3f}")